In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
import gymnasium as gym


from modules.agent_nets.modular import ModularAgentNetwork

from modules.embeddings.action_embedding import ActionEncoder

from modules.backbones.resnet import ResNetBackbone
from modules.backbones.conv import ConvBackbone

from modules.heads.value import ValueHead
from modules.heads.policy import PolicyHead
from modules.heads.reward import RewardHead
from modules.heads.to_play import ToPlayHead

from modules.world_models.muzero_world_model import MuzeroWorldModel
from modules.world_models.components.representation import Representation
from modules.world_models.components.dynamics import Dynamics

from agents.action_selectors.decorators import TemperatureSelector
from agents.action_selectors.selectors import CategoricalSelector
from agents.action_selectors.policy_sources import SearchPolicySource

# MuZero specific imports
from search.search_py.modular_search import ModularSearch
from replay_buffers.sequence import Sequence

# Learner imports
from agents.learner.base import UniversalLearner
from agents.learner.batch_iterators import SingleBatchIterator
from agents.learner.losses import (
    LossPipeline,
    ValueLoss,
    PolicyLoss,
    RewardLoss,
    ToPlayLoss,
)

from agents.learner.target_builders import (
    TargetBuilderPipeline,
    MCTSExtractor,
    SequencePadder,
    SequenceMaskBuilder,
    SequenceInfrastructureBuilder,
    TargetFormatter,
)

from agents.learner.losses.representations import (
    IdentityRepresentation,
    ClassificationRepresentation,
    ScalarRepresentation,
)

from agents.learner.losses.priorities import ExpectedValueErrorPriorityComputer

from utils.schedule import StepwiseSchedule

from replay_buffers.modular_buffer import BufferConfig, ModularReplayBuffer
from replay_buffers.processors import SequenceTensorProcessor, NStepUnrollProcessor
from replay_buffers.writers import SharedCircularWriter
from replay_buffers.samplers import UniformSampler
from replay_buffers.concurrency import TorchMPBackend

from agents.workers.actors import PettingZooActor
from agents.workers.tester import Tester
from agents.workers.tester import VsAgentTest
from agents.tictactoe_expert import TicTacToeBestAgent
import sys

sys.path.append("../../")
from configs.games.tictactoe import TicTacToeConfig

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ImportError: cannot import name 'create_muzero_buffer' from 'replay_buffers.buffer_factories' (/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/replay_buffers/buffer_factories.py)

In [ ]:
# --- Explicit Hyperparameters & Config ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 1. Environment via Config (solves NoneType obs_dim issue)
game_config = TicTacToeConfig()
env = game_config.env_factory()

SEARCH_BATCH_SIZE = 5
USE_VIRTUAL_MEAN = True
NUM_SIMULATIONS = 25
DISCOUNT_FACTOR = 0.99
UNROLL_STEPS = 5
TD_STEPS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
TOTAL_TRAINING_STEPS = 10000
TRAIN_STEPS_PER_EPISODE = 10
ACTION_EMBEDDING_DIM = 32
DIRICHLET_FRACTION = 0.25
DIRICHLET_ALPHA = 0.3
BUFFER_SIZE = 10000
TRANSFER_INTERVAL = 100
TEST_INTERVAL = 1000
NUM_WORKERS = 4
obs_dim = env.observation_space(agent="player_1").shape
num_actions = env.action_space(agent="player_1").n

print(f"Environment: TicTacToe | Obs Dim: {obs_dim} | Num Actions: {num_actions}")

Using device: cpu
Environment: TicTacToe | Obs Dim: (9, 3, 3) | Num Actions: 9


In [ ]:
# --- 1. Agent Network (World Model) ---
# Your ModularAgentNetwork must be refactored to explicitly accept World Model kwargs
action_encoder = ActionEncoder(
    action_space_size=num_actions,
    embedding_dim=ACTION_EMBEDDING_DIM,
    is_continuous=False,
    single_action_plane=True,
)


representation = Representation(
    backbone=ResNetBackbone(
        input_shape=obs_dim,
        filters=[24, 24, 24],
        kernel_sizes=[3, 3, 3],
        strides=[1, 1, 1],
        norm_type="batch",
    )
)
hidden_state_shape = representation.output_shape


dynamics = Dynamics(
    backbone=ResNetBackbone(
        input_shape=hidden_state_shape,
        filters=[24, 24, 24],
        kernel_sizes=[3, 3, 3],
        strides=[1, 1, 1],
        norm_type="batch",
    ),
    action_encoder=action_encoder,
    input_shape=hidden_state_shape,
    action_embedding_dim=ACTION_EMBEDDING_DIM,
)

reward_head = RewardHead(
    input_shape=hidden_state_shape,
    representation=ScalarRepresentation(),
    neck=ConvBackbone(
        input_shape=hidden_state_shape,
        filters=[16],
        kernel_sizes=[1],
        strides=[1],
        norm_type="batch",
    ),
)

to_play_head = ToPlayHead(
    input_shape=hidden_state_shape,
    num_players=2,
    neck=ConvBackbone(
        input_shape=hidden_state_shape,
        filters=[16],
        kernel_sizes=[1],
        strides=[1],
        norm_type="batch",
    ),
    representation=ClassificationRepresentation(num_classes=2),
)

world_model = MuzeroWorldModel(
    representation=representation,
    dynamics=dynamics,
    reward_head=reward_head,
    to_play_head=to_play_head,
    num_actions=num_actions,
)

prediction_backbone = ResNetBackbone(
    input_shape=hidden_state_shape,
    filters=[24, 24, 24],
    kernel_sizes=[3, 3, 3],
    strides=[1, 1, 1],
    norm_type="batch",
)

value_head = ValueHead(
    input_shape=hidden_state_shape,
    representation=ScalarRepresentation(),
    neck=ConvBackbone(
        input_shape=hidden_state_shape,
        filters=[16],
        kernel_sizes=[1],
        strides=[1],
        norm_type="batch",
    ),
)

policy_head = PolicyHead(
    input_shape=hidden_state_shape,
    neck=ConvBackbone(
        input_shape=hidden_state_shape,
        filters=[16],
        kernel_sizes=[1],
        strides=[1],
        norm_type="batch",
    ),
    representation=ClassificationRepresentation(num_classes=num_actions),
)


agent_network = ModularAgentNetwork(
    input_shape=obs_dim,
    num_actions=num_actions,
    components={
        "world_model": world_model,
        "prediction_backbone": prediction_backbone,
        "value_head": value_head,
        "policy_head": policy_head,
    },
    unroll_steps=UNROLL_STEPS,
    minibatch_size=BATCH_SIZE,
    atom_size=1,
).to(DEVICE)

# --- 2. Search Backend & Policy Source ---
search_engine = ModularSearch(
    device=DEVICE,
    num_actions=num_actions,
    bootstrap_method="v_mix",
    policy_extraction="visit",
    backprop_method="average",
    num_simulations=NUM_SIMULATIONS,
    discount_factor=DISCOUNT_FACTOR,
    dirichlet_alpha=DIRICHLET_ALPHA,
    dirichlet_fraction=DIRICHLET_FRACTION,
    use_virtual_mean=True,
    num_players=2,
)

policy_source = SearchPolicySource(
    search_engine=search_engine,
    agent_network=agent_network,
)


# TODO: temperature schedule
inner_selector = CategoricalSelector(exploration=True)
action_selector = TemperatureSelector(
    inner_selector=inner_selector,
    schedule=StepwiseSchedule(steps=[5], values=[1.0, 0.0]),
)

# --- 3. Replay Buffer ---
# TODO: dont use create_muzer_buffer
configs = [
    BufferConfig(
        "observations",
        shape=obs_dim,
        dtype=torch.float32,
        is_shared=True,
    ),
    BufferConfig("actions", shape=(), dtype=torch.float16, is_shared=True),
    BufferConfig("rewards", shape=(), dtype=torch.float32, is_shared=True),
    BufferConfig("values", shape=(), dtype=torch.float32, is_shared=True),
    BufferConfig("policies", shape=(num_actions,), dtype=torch.float32, is_shared=True),
    BufferConfig("to_plays", shape=(), dtype=torch.int16, is_shared=True),
    BufferConfig("chances", shape=(1,), dtype=torch.int16, is_shared=True),
    BufferConfig("game_ids", shape=(), dtype=torch.int64, is_shared=True),
    BufferConfig("ids", shape=(), dtype=torch.int64, is_shared=True),
    BufferConfig("training_steps", shape=(), dtype=torch.int64, is_shared=True),
    BufferConfig("terminated", shape=(), dtype=torch.bool, is_shared=True),
    BufferConfig("truncated", shape=(), dtype=torch.bool, is_shared=True),
    BufferConfig("dones", shape=(), dtype=torch.bool, is_shared=True),
    BufferConfig("legal_masks", shape=(num_actions,), dtype=torch.bool, is_shared=True),
]

input_processor = SequenceTensorProcessor(
    num_actions, 2, {"player_1": 0, "player_2": 1}
)

output_processor = NStepUnrollProcessor(
    UNROLL_STEPS,
    TD_STEPS,
    DISCOUNT_FACTOR,
    num_actions,
    2,
    BUFFER_SIZE,
)

replay_buffer = ModularReplayBuffer(
    max_size=BUFFER_SIZE,
    batch_size=BATCH_SIZE,
    buffer_configs=configs,
    input_processor=input_processor,
    output_processor=output_processor,
    writer=SharedCircularWriter(max_size=BUFFER_SIZE),
    sampler=UniformSampler(),
    backend=TorchMPBackend(),
)

# --- 4. Optimizer and Learner Components ---
optimizer = {
    "default": torch.optim.Adam(agent_network.parameters(), lr=LEARNING_RATE, eps=1e-5)
}
# Extract representations from heads
val_rep = agent_network.components["value_head"].representation
pol_rep = agent_network.components["policy_head"].representation
rew_rep = agent_network.components["world_model"].reward_head.representation
tp_rep = agent_network.components["world_model"].to_play_head.representation

modules = [
    ValueLoss(
        device=DEVICE,
        loss_fn=nn.functional.mse_loss,
        loss_factor=1.0,
    ),
    PolicyLoss(
        device=DEVICE,
        loss_fn=nn.functional.cross_entropy,
        loss_factor=1.0,
    ),
    RewardLoss(
        device=DEVICE,
        loss_fn=nn.functional.mse_loss,
        loss_factor=1.0,
    ),
    ToPlayLoss(
        device=DEVICE,
        # loss_fn=nn.functional.cross_entropy, # TODO: update this to have a loss_fn
        loss_factor=1.0,
    ),
]


priority_computer = ExpectedValueErrorPriorityComputer(value_representation=val_rep)
loss_pipeline = LossPipeline(
    modules=modules,
    priority_computer=priority_computer,
    minibatch_size=BATCH_SIZE,
    unroll_steps=UNROLL_STEPS,
    num_actions=num_actions,
    atom_size=1,
    # Centralized mapping for MuZero heads
    representations={
        "values": val_rep,
        "policies": pol_rep,
        "rewards": rew_rep,
        "to_plays": tp_rep,
    },
)

builders = [
    MCTSExtractor(),
    SequencePadder(unroll_steps=UNROLL_STEPS),
    SequenceMaskBuilder(),
    SequenceInfrastructureBuilder(unroll_steps=UNROLL_STEPS),
    TargetFormatter({"values": val_rep, "policies": pol_rep, "rewards": val_rep}),
]

target_builder = TargetBuilderPipeline(builders)

learner = UniversalLearner(
    agent_network=agent_network,
    device=DEVICE,
    num_actions=num_actions,
    observation_dimensions=obs_dim,
    observation_dtype=torch.float32,
    loss_pipeline=loss_pipeline,
    optimizer=optimizer,
    target_builder=target_builder,
    lr_scheduler={},
    callbacks=[],
    clipnorm=5.0,
)

print("All components initialized successfully!")

All components initialized successfully!


In [ ]:
import torch.multiprocessing as mp
from agents.executors.torch_mp_executor import TorchMPExecutor

# Multiprocessing in Jupyter typically requires the spawn method
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

# 1. Share model memory so parallel workers see weight updates implicitly
agent_network.share_memory()

# 3. Setup TorchMP Executor and detect appropriate worker class (GymActor vs PettingZooActor)
executor = TorchMPExecutor()

# 4. Define arguments for BaseActor.__init__
launch_args = (
    game_config.env_factory,
    agent_network,
    action_selector,
    replay_buffer,
    2,
    torch.device("cpu"),  # Workers typically run MCTS on CPU to avoid CUDA IPC overhead
    "muzero_worker",
)

launch_kwargs = {"search_engine": search_engine}

# Launch the parallel workers
executor.launch(PettingZooActor, launch_args, num_workers=NUM_WORKERS, **launch_kwargs)
# TODO: add test vs agent test type etc.

launch_args = (
    game_config.env_factory,
    agent_network,
    action_selector,
    None,  # replay_buffer (Tester doesn't use it)
    2,
    torch.device("cpu"),
    "tester",
    [
        VsAgentTest(
            name=f"vs_expert_p0",
            num_trials=100,
            opponent=TicTacToeBestAgent(),
            player_idx=0,
        ),
        VsAgentTest(
            name=f"vs_expert_p1",
            num_trials=100,
            opponent=TicTacToeBestAgent(),
            player_idx=1,
        ),
    ],
)

executor.launch(Tester, launch_args, num_workers=1, **launch_kwargs)
print(
    f"Starting explicit MuZero training loop with {NUM_WORKERS} parallel TorchMP workers..."
)

train_step = 0
from stats.stats import StatTracker

stats = StatTracker(
    "muzero",
)

while train_step < TOTAL_TRAINING_STEPS:
    # ==========================================
    # PHASE 1: Data Collection via Executor
    # ==========================================
    # The workers handle the env loop natively (including PettingZoo turn logic).
    # collect_data() blocks until the workers complete sequences and push them to the buffer.
    # It returns the episode statistics.
    results, collect_stats = executor.collect_data(
        min_samples=None, worker_type=PettingZooActor
    )

    # 2. Log collection stats
    for key, val in collect_stats.items():
        stats.append(key, val)

    # 3. Learning step
    if replay_buffer.size >= BATCH_SIZE:
        iterator = SingleBatchIterator(replay_buffer, DEVICE)
        for step_metrics in learner.step(iterator):
            if train_step % 1000 == 0:
                print(step_metrics)

        train_step += 1

        # 4. Update workers (if needed)
        if train_step % TRANSFER_INTERVAL == 0:
            executor.update_weights(agent_network.state_dict())

        # 6. Periodic testing
        if train_step % TEST_INTERVAL == 0:
            executor.request_work(Tester)

            # If local, run synchronously now
            results, _ = executor.collect_data(min_samples=None, worker_type=Tester)
            print(results)
# Cleanup parallel processes
executor.stop()
print("MuZero Parallel Training complete!")

Starting explicit MuZero training loop with 4 parallel TorchMP workers...


/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report 

Initializing stat 'score' with subkeys None
Initializing stat 'episode_length' with subkeys None
Initializing stat 'num_episodes' with subkeys None
Initializing stat 'actor_fps' with subkeys None
{'ValueLoss': 1.0475215911865234, 'approx_kl': 0.7140247225761414, 'PolicyLoss': 2.2703094482421875, 'RewardLoss': 0.22710692882537842, 'ToPlayLoss': 0.7461491227149963, 'loss_pipeline_latency_ms': 2.524124982301146, 'learner_throughput': 280.1724038685213, 'loss': 4.2910871505737305}
{'ValueLoss': 0.7518905401229858, 'approx_kl': 0.5896511673927307, 'PolicyLoss': 2.2094216346740723, 'RewardLoss': 0.12457692623138428, 'ToPlayLoss': 0.7705192565917969, 'loss_pipeline_latency_ms': 0.8596659754402936, 'learner_throughput': 521.8828411261518, 'loss': 3.8564085960388184}
{'ValueLoss': 0.5757431983947754, 'approx_kl': 0.7481322884559631, 'PolicyLoss': 2.2158539295196533, 'RewardLoss': 0.18216456472873688, 'ToPlayLoss': 0.01050360593944788, 'loss_pipeline_latency_ms': 1.4902920229360461, 'learner_thr